# Libraries

In [1]:
import os
import sys

root_models = '/kaggle/input/models/rickmark369/symbolic-regression-thesis-magistrale-unipi/other/default/18'

sys.path.insert(0, root_models)
sys.path.insert(0, "/kaggle/input/datasets")

root_dataset = '/kaggle/input/datasets/rickmark369/tesi-magistrale-unipi/'
root_models  = root_models + '/'


root_dataset = ""
root_models = ""

In [2]:
#%pip install lark graphviz xicorpy pmlb

In [ ]:
import pmlb
import pandas as pd
import numpy as np
import time
import yaml
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
import json

import LearningStrategy as ls
from Model import SymbolicRegression
import Errors as er

# Prepare datasets

In [4]:
cavalab_bb_results = pd.read_feather(root_dataset + 'InformationSOTA/srbench-master/results/black-box_results.feather')
cavalab_gt_results = pd.read_feather(root_dataset + 'InformationSOTA/srbench-master/results/ground-truth_results.feather')

bb_dataset_names = [i for i in cavalab_bb_results["dataset"].unique()]
gt_dataset_names = [i for i in cavalab_gt_results["dataset"].unique()]

In [5]:
#Calculate the seeds used in every datasets
dict_seeds = {}

for name in bb_dataset_names:
    dict_seeds[name] = set([x for x in cavalab_bb_results.loc[cavalab_bb_results["dataset"] == name, "random_state"].unique()])
    
for name in gt_dataset_names:
    dict_seeds[name] = set([x for x in cavalab_gt_results.loc[cavalab_gt_results["dataset"] == name, "random_state"].unique()])

total_set = set([x for x in cavalab_bb_results["random_state"].unique()]).union(set([x for x in cavalab_gt_results["random_state"].unique()]))

for key in dict_seeds.keys():
    
    total_set = total_set.intersection(dict_seeds[key])
    
#print(total_set)

In [6]:
def split(x, perc1, perc2):
    train_len = int(perc1*len(x))
    val_len = int((perc1+perc2)*len(x))
    
    #TR, VL, TS
    return x[:train_len], x[train_len:val_len], x[val_len:]

def is_prefix(name, prefix):
    return name[:len(prefix)] == prefix

def adapt_name(name):
    
    expection_names = {
        "feynman_I_48_2": "feynman_I_48_20",
                       }
    
    if(name in expection_names.keys()): name = expection_names[name]
    
    if(is_prefix(name, "feynman_test")): return name[len('feynamn_'):]
    elif(is_prefix(name, "feynman")): return name[len('feynamn_'):].replace('_', '.')
    elif(is_prefix(name, "strogatz")): return name.replace('strogatz', 'd')



#Si ridà direttamente il dataset splittato: si fa uno split hold out del 75 di training set e 25% di test set (lo dice alla fine della sezione 3.2 del paper Call for action, tipo preprint).
#Da capire il seed quali sono.
def get_dataset(name):

    if(name in bb_dataset_names):
    
        x, y = pmlb.fetch_data(name, return_X_y=True)
        
        x = x.T

        result = {"x_dict": {f"X{i}": x[i] for i in range(x.shape[0])}, "y": y, "formula": None}
    
    elif(name in gt_dataset_names):
        
        adapted_name = adapt_name(name)
        
        formulas = pd.read_csv(root_dataset + "InformationSOTA/Equations.csv")
        
        try: formula_founded = formulas.loc[formulas["Filename"] == adapted_name, "Formula"].tolist()[0]
        except IndexError: formula_founded = None
        
        if(is_prefix(name, "feynman_test")): file = root_dataset + f"InformationSOTA/Feynman_bonus_with_units/{adapted_name}"
        elif(is_prefix(name, "feynman")): file =    root_dataset + f"InformationSOTA/Feynman_with_units/{adapted_name}"
        elif(is_prefix(name, "strogatz")): file =   root_dataset + f"InformationSOTA/ode-strogatz-master/{adapted_name}.txt"
        
        if(is_prefix(name, "feynman")):
            
            feature_names = [x["name"] for x in yaml.safe_load(open(root_dataset + f"InformationSOTA/pmlb-master/datasets/{name}/metadata.yaml", "r"))["features"]]
            
            table = np.loadtxt(file).T
            result = {"x_dict": {name: table[i] for name, i in zip(feature_names, range(table.shape[0]-1))}, "y": table[-1], "formula": formula_founded}
        
        else:
            
            table = pd.read_csv(file)
            result = {"x_dict": {l: table[l].to_numpy() for l in table.columns.tolist() if l != "label"}, "y": table["label"].to_numpy(), "formula": formula_founded}
            
        
    else: raise ValueError(f"Dataset {name} are not aviable")
     
    if(np.any(~np.isfinite(result["y"])) or any(np.any(~np.isfinite(result["x_dict"][k])) for k in result["x_dict"].keys())):
        raise ValueError("Some data are not finite value, the dataset is not valid")
    
    
    #Prepare the hyper-parameters choosed for this experiments
    
    shuffle_seed = 860  #Form the 5 universal seed, we choose 860

    #Prepare the slots for split
    result["x_dict train"] = {}
    result["x_dict val"] = {}
    result["x_dict test"] = {}
    
    #Prepare the shuffler
    shuffler = np.random.default_rng(seed = shuffle_seed)
    
    indexes = np.arange(len(result["y"]))
    shuffler.shuffle(indexes)

    TR_perc = 0.60
    VL_perc = 0.15
    
    #Split all X
    for key in result["x_dict"].keys():
        result["x_dict train"][key], result["x_dict val"][key], result["x_dict test"][key] = split(result["x_dict"][key][indexes], TR_perc, VL_perc)
    
    #Split Y
    result["y train"], result["y val"], result["y test"] = split(result["y"][indexes], TR_perc, VL_perc)
    
    #Delete old Y and all X
    del result["x_dict"]
    del result["y"]

    return result

In [7]:
def ParallelExecution(threads, f_to_execute, list_to_apply_execution):
    return list(ThreadPoolExecutor(max_workers=threads).map(f_to_execute, list_to_apply_execution))

In [8]:
def calculate_dimension(dataset_name):

    print(dataset_name, "execution")

    try:

        #Extract inputs and output accordind to the dataset name
        dataset = get_dataset(dataset_name)
        TR_inputs_train = dataset["x_dict train"]
        TR_output_train = dataset["y train"]
        VL_output_train = dataset["y val"]
        TS_output_test = dataset["y test"]
    
        return {'dataset': dataset_name,
                'features': len(TR_inputs_train.keys()),
                'TR points': len(TR_output_train),
                'VL points': len(VL_output_train),
                'TS points': len(TS_output_test)}

    except: return {'dataset': dataset_name}

dataset_names = bb_dataset_names + gt_dataset_names

results = []

#results = ParallelExecution(4, calculate_dimension, dataset_names)
#json.dump(results, open(f"DatasetDimension.json", "w", encoding="utf-8"), indent=4, ensure_ascii=False)

# Prepare tools for training

In [9]:
class Counter:
        
    def __init__(self, i):
        self.__i = i
        self.__r = 0
            
    def __call__(self):
        self.__r += 1
        return self.__r > self.__i


def execute(dataset_name):

    #try:

    #Extract inputs and output accordind to the dataset name
    dataset = get_dataset(dataset_name)
    TR_inputs_train = dataset["x_dict train"]
    TR_output_train = dataset["y train"]

    counter = Counter(250)
            
    tree = SymbolicRegression.string_to_tree("0")
            
    metric_results = tree.learn(TR_inputs_train,
                                TR_output_train,
                                stop_criteria = lambda ls: counter(),
                                metric_functions = {"tree": lambda model: str(model), "timestamp": lambda model: time.time()})
            
    #if(not os.path.exists("BenchmarkResults")): os.mkdir("BenchmarkResults")
                
    #json.dump(metric_results, open(f"BenchmarkResults/{dataset_name}.json", "w", encoding="utf-8"), indent=4, ensure_ascii=False)
    #print(f"Results of dataset {dataset_name} saved!")  
    
    #except: print(f"Error on training model with {dataset_name} dataset")

# Experiments

In [10]:
dataset_to_do = [
  
#'feynman_I_6_2a',
#'feynman_I_15_3t',
#'feynman_I_18_14',
#'feynman_I_26_2',
#'feynman_I_40_1',
#'feynman_I_44_4 ',
#'feynman_II_11_27',
#'feynman_III_15_12',
#'feynman_test_10',
#'feynman_test_11',

#"1027_ESL",
#"1028_SWD",
#"1029_LEV",
#"1030_ERA",
"1089_USCrime",
"1096_FacultySalaries",
"1191_BNG_pbc",
"1193_BNG_lowbwt",
"1196_BNG_pharynx",
"1199_BNG_echoMonths",
"1201_BNG_breastTumor",
"1203_BNG_pwLinear",
"1595_poker",
"192_vineyard",
"195_auto_price",
"197_cpu_act",
"201_pol",
"207_autoPrice",
"210_cloud",
"215_2dplanes",
"218_house_8L",
"225_puma8NH",
"227_cpu_small",
"228_elusage",
"229_pwLinear",
"230_machine_cpu",
"294_satellite_image",
"344_mv",
"4544_GeographicalOriginalofMusic",
"485_analcatdata_vehicle",
"503_wind",
"505_tecator",
"519_vinnie",
"522_pm10",
"523_analcatdata_neavote",
"527_analcatdata_election2000",
"529_pollen",
"537_houses",
"542_pollution",
"547_no2",
"556_analcatdata_apnea2",
"557_analcatdata_apnea1",
"560_bodyfat",
"561_cpu",
"562_cpu_small",
"564_fried",
"573_cpu_act",
"574_house_16H",
"579_fri_c0_250_5",
"581_fri_c3_500_25",
"582_fri_c1_500_25",
"583_fri_c1_1000_50",
"584_fri_c4_500_25",
"586_fri_c3_1000_25",
"588_fri_c4_1000_100",
"589_fri_c2_1000_25",
"590_fri_c0_1000_50",
"591_fri_c1_100_10",
"592_fri_c4_1000_25",
"593_fri_c1_1000_10",
"594_fri_c2_100_5",
"595_fri_c0_1000_10",
"596_fri_c2_250_5",
"597_fri_c2_500_5",
"598_fri_c0_1000_25",
"599_fri_c2_1000_5",
"601_fri_c1_250_5",
"602_fri_c3_250_10",
"603_fri_c0_250_50",
"604_fri_c4_500_10",
"605_fri_c2_250_25",
"606_fri_c2_1000_10",
"607_fri_c4_1000_50",
"608_fri_c3_1000_10",
"609_fri_c0_1000_5",
"611_fri_c3_100_5",
"612_fri_c1_1000_5",
"613_fri_c3_250_5",
"615_fri_c4_250_10",
"616_fri_c4_500_50",
"617_fri_c3_500_5",
"618_fri_c3_1000_50",
"620_fri_c1_1000_25",
"621_fri_c0_100_10",
"622_fri_c2_1000_50",
"623_fri_c4_1000_10",
"624_fri_c0_100_5",
"626_fri_c2_500_50",
"627_fri_c2_500_10",
"628_fri_c3_1000_5",
"631_fri_c1_500_5",
"633_fri_c0_500_25",
"634_fri_c2_100_10",
"635_fri_c0_250_10",
"637_fri_c1_500_50",
"641_fri_c1_500_10",
"643_fri_c2_500_25",
"644_fri_c4_250_25",
"645_fri_c3_500_50",
"646_fri_c3_500_10",
"647_fri_c1_250_10",
"648_fri_c1_250_50",
"649_fri_c0_500_5",
"650_fri_c0_500_50",
"651_fri_c0_100_25",
"653_fri_c0_250_25",
"654_fri_c0_500_10",
"656_fri_c1_100_5",
"657_fri_c2_250_10",
"658_fri_c3_250_25",
"659_sleuth_ex1714",
"663_rabe_266",
"665_sleuth_case2002",
"666_rmftsa_ladata",
"678_visualizing_environmental",
"687_sleuth_ex1605",
"690_visualizing_galaxy",
"695_chatfield_4",
"706_sleuth_case1202",
"712_chscase_geyser1",
"banana",
"titanic",

'strogatz_bacres1',
'strogatz_bacres2',
'strogatz_barmag1',
'strogatz_barmag2',
'strogatz_glider1',
'strogatz_glider2',
'strogatz_lv1',
'strogatz_lv2',
'strogatz_predprey1',
'strogatz_predprey2',
'strogatz_shearflow1',
'strogatz_shearflow2',
'strogatz_vdp1',
'strogatz_vdp2',
]

dataset_done = []

In [11]:
import os

names = ["'" + i.split(".")[0] + "'" for i in os.listdir("BenchmarkResults")]

print("[" + ",\n".join(names) + "]")

[]


In [ ]:
for x in dataset_to_do:
    if(x not in dataset_done):
        print(x)
        execute(x)

1089_USCrime
tree 0.0
timestamp 1791922910.7556326
tree 0.0
timestamp 1791922911.3172252
Iteration done 1 ----------------------------------------------------------------------------------------------------
tree 0.0
timestamp 1791922911.669317
Iteration done 2 ----------------------------------------------------------------------------------------------------
tree 0.0
timestamp 1791922911.9276276
Iteration done 3 ----------------------------------------------------------------------------------------------------
tree 0.0
timestamp 1791922912.371186
Iteration done 4 ----------------------------------------------------------------------------------------------------
tree sin(X2)
timestamp 1791922912.600954
Iteration done 5 ----------------------------------------------------------------------------------------------------
tree cos(X2)
timestamp 1791922912.8406122
Iteration done 6 ----------------------------------------------------------------------------------------------------
tree sin

# Calculate errors on datasets

In [ ]:
def linear_extension(X, Y):
    
    mean_X = np.mean(X)
    mean_Y = np.mean(Y)
        
    m = np.sum((X - mean_X)*(Y - mean_Y)) / np.sum((X - mean_X)**2)
    q = mean_Y - (m*mean_X)
    
    return m*X + q



dataset_already_done = [x.split(".")[0] for x in os.listdir(root_models + '/BenchmarkResults/')]

error_metrics = {"MSE": lambda y_model, y_true: er.MSE(y_model, y_true),
                 "MAE": lambda y_model, y_true: er.MAE(y_model, y_true),
                 "R2": lambda  y_model, y_true: er.R2(y_model, y_true)}



def apply(results, tree, TR_inputs, TR_output, label):

    try: output_model = tree.get_root().forward_calculation_memory_sub_tree(TR_inputs).get_memory("forward")
    except: output_model = np.nan
    
    output_model_lin_ext = linear_extension(output_model, TR_output)
    
    for metric in error_metrics.keys(): 
        
        internal_label = f"{label} {metric}"
        if(internal_label not in results): results[internal_label].append(error_metrics[metric](output_model, TR_output))
        
        internal_label = f"{label} {metric} lin. ext."
        if(internal_label not in results): results[internal_label].append(error_metrics[metric](output_model_lin_ext, TR_output))
    
    

def calculate_error(dataset_name):

    print(f"Calculate errors about {dataset_name} dataset")

    experiment = json.load(open(root_models + f"/BenchmarkResults/{dataset_name}.json", "r"))

    #Extract inputs and output accordind to the dataset name
    dataset = get_dataset(dataset_name)
    
    TR_inputs = dataset["x_dict train"]
    TR_output = dataset["y train"]
    
    VL_inputs = dataset["x_dict val"]
    VL_output = dataset["y val"]

    TS_inputs = dataset["x_dict test"]
    TS_output = dataset["y test"]
    
    errors = {"Nodes" : []}

    for step in experiment["results"]["tree"]:
        
        tree = SymbolicRegression.string_to_tree(step)
        
        errors["Nodes"].append(len(tree.get_root().get_all_subtree_nodes()))

        apply(errors, tree, TR_inputs, TR_output, "TR")
        apply(errors, tree, VL_inputs, VL_output, "VL")
        apply(errors, tree, TS_inputs, TS_output, "TS")
      
    if(not os.path.exists("BenchmarkCompleteResults")): os.mkdir("BenchmarkCompleteResults")

    experiment["results"] |= errors
            
    json.dump(experiment, open(f"BenchmarkCompleteResults/{dataset_name}.json", "w", encoding="utf-8"), indent=4, ensure_ascii=False)
    print(f"Results of adding errors {dataset_name} saved!")   

#ParallelExecution(4, calculate_error, dataset_already_done)